# 13 · 把三个工具做成标准接口（MCP）

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → [13 标准接口 MCP ✅] → 14 起：固定流程、agent …
```

> **这一章的代码在合并请求 #15 里。** 这一章不调大模型，一分钱不花。

## 背景：工具做好了，别的程序怎么用？

现在想用 `rank_diseases`，得在我们的项目里写 Python：

```python
from raredx.tools.rank_diseases import RankDiseases
rank_diseases = RankDiseases(hpo)
rank_diseases(["HP:0000786", "HP:0008209"])
```

可是要用工具的，往往不是我们自己的代码，而是 Claude Code、Claude Desktop 这样的 AI 应用，或者 LangGraph 写的 agent。以前每个应用接工具的方式都不一样，工具要给每个应用各包一遍。

**MCP**（Model Context Protocol，模型上下文协议）就是为这个定的标准。Anthropic 2024 年底提出，现在 OpenAI、Google、微软的产品也都支持。打个比方，它是工具的 **USB 接口**：以前每种设备一个插头，现在统一成 USB，设备做一次，哪台电脑都能插。工具按 MCP 做一次，哪个 AI 应用都能用。

这一章把三个工具做成一个 MCP 服务，然后像真实的 AI 应用那样，从外面连上去调用。

## 这一章做的 5 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 把三个工具挂到 MCP 服务上 | 官方 SDK 的 `MCPServer` | 不自己写协议 |
| 2 | 像真实应用一样连上去 | stdio：应用启动我们的命令 | 证明真的能用 |
| 3 | 模型看到的"说明书" | 从代码自动生成 | 说明和代码永远一致 |
| 4 | 出错时模型看到什么 | 工具错误、自动拒绝坏参数 | 模型能自己改正 |
| 5 | 接到 Claude Code | `.mcp.json` | 团队打开项目就能用 |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 把三个工具挂到 MCP 服务上

### 在干什么
写一个 MCP 服务，把三个工具挂上去，并加一条命令 `raredx mcp` 来启动它。

### 背景
协议本身很细：消息格式、握手、每个字段怎么写。自己从头实现，容易出错，也没人帮你维护。

### 用了什么技术，为什么
用 MCP 的**官方 Python SDK**（`mcp` 2.3 版，MIT 许可证）里的 `MCPServer`。我们的服务代码（[`src/raredx/mcp_server.py`](../../src/raredx/mcp_server.py)）只有几十行，只做三件事：

| 做的事 | 为什么 |
|---|---|
| 把三个工具原样挂上去 | 工具还是第 10–12 章那几个，测试过的，服务里不加新逻辑 |
| 每个工具标上"只读"和标题 | 应用看到"只读"，就知道调用它不会改任何东西、不会联网，可以放心让模型调 |
| 把"病不存在"转成模型看得见的错误 | 第 4 步细讲 |

工具的说明、参数、返回格式，SDK 直接从工具的代码里读，就是第 10–12 章写的那些说明。

> **一个真实踩过的坑。** 第一次挂上去，所有调用都失败了：SDK 把工具方法里的 `self`（Python 方法自带的第一个参数）也当成了要模型填的参数，说明书里多出一个 `self`。修法是把去掉 `self` 的参数表交给 SDK，一行代码。有一条测试守着：说明书里只有 `hpo_ids` 和 `top_k`。

### 公司里什么情况怎么做
- **用官方 SDK**，不自己实现协议
- **服务层不加业务逻辑**：只做"转接"。逻辑都在工具里，工具单独测过

### 执行
看看这条新命令：

In [ ]:
run("uv run raredx mcp --help")

### 结果怎么看
- 默认是 `stdio`：给"会启动这条命令的应用"用，下一步就是这种
- 另一种 `streamable-http`：把工具挂在一个网址上，给远程的程序用（以后部署时用）

### 面试怎么说
> "The tools are served over MCP with the official Python SDK. The server holds no logic: it registers the tested tools as they are, marks them read-only, and turns errors the model can fix into tool errors."

---

## 第 2 步 · 像真实的 AI 应用一样连上去

### 在干什么
用 MCP 官方客户端启动 `raredx mcp`，连上去，列出工具、调用工具。Claude Code 这类应用就是这么做的。

### 背景
单元测试证明代码对，但"别的程序真的能用"要亲眼看一次：另起一个进程，只通过标准协议说话，不 import 我们的任何代码。

### 用了什么技术，为什么
**stdio**：应用把我们的命令当成一个子进程启动，通过它的标准输入、输出来回传消息。不开端口，不联网。

```
AI 应用（Claude Code、agent）
   │  启动子进程：uv run raredx mcp
   ▼
我们的 MCP 服务 ── 加载 HPO，等消息
   ▲  "有哪些工具？" "调用 rank_diseases，参数是……"
   │  回答：工具列表、调用结果（JSON）
```

### 公司里什么情况怎么做
- **本地工具用 stdio**：每个用户的电脑上启动一份，数据不出这台电脑
- **共享服务用 HTTP**：部署在服务器上，大家连同一个网址（以后"做成服务"那一步）

### 执行
连上去，调用两个工具（服务端的日志写到一个临时文件里，最后删掉）：

In [ ]:
import json, time
from mcp import Client, StdioServerParameters, stdio_client

PERRAULT = ["HP:0000786", "HP:0008209", "HP:0008619", "HP:0001256", "HP:0001263"]  # 第 10 章那位病人
server = StdioServerParameters(command="uv", args=["run", "raredx", "mcp"], cwd=str(ROOT))
log = ROOT / "mcp-server.log"

started = time.perf_counter()
with log.open("w") as errlog:
    async with Client(stdio_client(server, errlog=errlog)) as client:
        print(f"连上了，用时 {time.perf_counter() - started:.1f} 秒（启动程序 + 加载 HPO）\n")
        tools = (await client.list_tools()).tools
        for t in tools:
            print(f"工具：{t.name:<15} 标题：{t.title}")

        ranking = await client.call_tool("rank_diseases", {"hpo_ids": PERRAULT, "top_k": 3})
        print("\nrank_diseases 前 3 名：")
        for m in ranking.structured_content["matches"]:
            print(f"   {m['disease_id']}  {m['disease_name']}")

        report = await client.call_tool("check_match", {"hpo_ids": PERRAULT, "disease_id": "OMIM:621101"})
        r = report.structured_content
        print(f"\ncheck_match（Perrault 7 型）：解释了 {len(r['explained'])} / {r['n_findings']} 个症状")
log.unlink()

### 结果怎么看
- 连接用了几秒：启动程序、加载 HPO 都在这里，之后每次调用都很快
- 三个工具，每个带一个给界面显示的标题
- `rank_diseases` 前 3 名是那几个耳聋病，`check_match` 说 Perrault 7 型解释了 5/5 个症状：**和第 10、12 章直接调用的结果一模一样**。中间隔了一个进程、一层协议，结果不能变（有一条测试专门比对这一点）

### 面试怎么说
> "A host launches `raredx mcp` as a subprocess and talks to it over stdin and stdout; results through the protocol are tested to be identical to direct calls."

---

## 第 3 步 · 模型看到的"说明书"

### 在干什么
看看应用从服务拿到的工具说明。模型决定"调不调、怎么调"，读的就是这个。

### 背景
第 10 章说过：模型不读代码，只看工具的名字、说明、每个参数的说明。那这些说明长什么样？

### 用了什么技术，为什么
SDK 从工具的代码里**自动生成**一份标准格式（JSON Schema）的说明书：说明来自工具的文档字符串，参数来自它的类型和限制。好处是说明书和代码**永远一致**：代码改了，说明书跟着变，不会"说一套、做一套"。

### 公司里什么情况怎么做
- **说明书由代码生成**，不手写第二份
- **改工具的说明要走评审**，和改提示词一样（第 10 章）

### 执行

In [ ]:
with log.open("w") as errlog:
    async with Client(stdio_client(server, errlog=errlog)) as client:
        rank = next(t for t in (await client.list_tools()).tools if t.name == "rank_diseases")
log.unlink()

print("说明：", rank.description, "\n")
print("参数（JSON Schema）：")
print(json.dumps(rank.input_schema["properties"], indent=2, ensure_ascii=False))
print("\n必填：", rank.input_schema["required"], "  只读：", rank.annotations.read_only_hint)

### 结果怎么看
- **说明**：第 10 章给工具写的那段英文，原封不动
- **`hpo_ids`**：一个列表，每个元素必须匹配 `^HP:\d{7}$`（HP: 加 7 位数字），至少 1 个
- **`top_k`**：整数，1 到 50，默认 10
- **必填**只有 `hpo_ids`；**只读**是 true

这些都是从工具的代码里生成的，我们没有另写一份。

### 面试怎么说
> "The schema a model sees is generated from the tool's signature and docstring — descriptions, types, limits, defaults — so documentation and behaviour cannot drift apart."

---

## 第 4 步 · 出错时模型看到什么

### 在干什么
故意传错，看模型会收到什么。

### 背景
模型会犯错：编号写错格式、编一个不存在的病号。好的工具要让它**知道错在哪、怎么改**，它才能自己重试。

### 用了什么技术，为什么
两种错误，两种处理：

| 错误 | 例子 | 谁拦下 | 模型收到什么 |
|---|---|---|---|
| 参数格式不对 | 症状写成 `"deafness"` | SDK，工具还没运行就拒绝 | "必须匹配 HP: 加 7 位数字" |
| 格式对，但病不存在 | `OMIM:123456` | 我们的工具 | "这个病不存在，请用 rank_diseases 返回的编号" |

第二种要专门处理：SDK 默认会把工具里的报错**藏起来**，只说"工具执行出错"。这是为了安全：程序内部的出错信息可能暴露服务器细节，不能随便给外面看。所以我们**明确标出**"病不存在"这一种是模型能改正的，它的原话才会传给模型。

### 公司里什么情况怎么做
- **分清两类错**：模型能改的（告诉它原因）；程序自己的 bug（只记日志，不外泄细节）
- **错误信息写成"下一步怎么做"**，而不只是"错了"

### 执行

In [ ]:
with log.open("w") as errlog:
    async with Client(stdio_client(server, errlog=errlog)) as client:
        bad_format = await client.call_tool("rank_diseases", {"hpo_ids": ["deafness"]})
        no_such_disease = await client.call_tool("lookup_disease", {"disease_id": "OMIM:123456"})
log.unlink()

for name, result in (("格式不对", bad_format), ("病不存在", no_such_disease)):
    print(f"{name}：出错 = {result.is_error}")
    print("   模型收到：", "\n             ".join(result.content[0].text.splitlines()[:3]))
    print()

### 结果怎么看
- 两次都是 `出错 = True`，但调用本身没有崩：错误作为一条**普通的回答**交回给模型
- 格式不对：模型看到的错误里写着要匹配的格式
- 病不存在：模型看到我们写的那句话，知道下一步该用 `rank_diseases` 给出的编号

### 面试怎么说
> "Errors are part of the conversation: malformed arguments are refused against the schema before the tool runs, and an unknown disease comes back as a tool error whose message tells the model how to recover. Unexpected exceptions stay in the server log."

---

## 第 5 步 · 接到 Claude Code（以及别的应用）

### 在干什么
让 Claude Code 能直接用我们的工具。

### 背景
不同的应用，告诉它"怎么启动这个服务"的地方不一样，但内容都是同一条命令：`uv run raredx mcp`。

### 用了什么技术，为什么
项目根目录放一个 **`.mcp.json`**，这是 Claude Code 读的项目级配置。它进 git，所以团队里谁打开这个项目，都有同样的工具。

怎么用：
1. 在 `rare-disease-agent` 文件夹里打开 Claude Code（终端里进入这个文件夹运行 `claude`，或用 VS Code 打开这个文件夹）
2. 它会问要不要启用 `raredx` 这个服务；同意后，输入 `/mcp` 能看到三个工具
3. 直接问："病人有原发性闭经、卵巢功能不全、双侧感音神经性耳聋，用 raredx 的工具查一下可能是什么病"

其他应用也是同一条命令。比如 Claude Desktop，在它的配置文件里写：

```json
{"mcpServers": {"raredx": {"command": "/绝对路径/uv",
  "args": ["run", "--directory", "/绝对路径/rare-disease-agent", "raredx", "mcp"]}}}
```

（Claude Desktop 不是从项目文件夹启动的，所以路径要写全。）

### 公司里什么情况怎么做
- **项目级配置进 git**：新同事打开项目就能用，不用一个个教
- **用的时候也留记录**：每次有应用调用我们的工具，都记进 MLflow 的 `mcp-server` 实验，出了问题能查

### 执行
看看配置文件，再看刚才这几步的调用有没有记进 MLflow：

In [ ]:
run("cat .mcp.json")

from raredx.config import settings  # raredx first: it sets MLflow's switches before mlflow loads
import mlflow

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
experiment = mlflow.get_experiment_by_name("mcp-server")
traces = mlflow.search_traces(locations=[experiment.experiment_id], order_by=["timestamp_ms DESC"],
                              return_type="list", max_results=1000)
print(f"MLflow 实验 mcp-server 里一共 {len(traces)} 条调用记录，最近 3 条：")
for t in traces[:3]:
    print("  ", t.data.spans[0].name, "|", t.info.state.value)

### 结果怎么看
- `.mcp.json` 里就是那条启动命令
- MLflow 的 `mcp-server` 实验里有刚才的调用：最近一条是第 4 步"病不存在"那次（`ERROR`），再往前两条是第 2 步的两次正常调用（`OK`）。出错的调用也有记录，出了问题能查到模型当时传了什么
- 第 4 步"格式不对"那次**没有记录**：SDK 在工具运行之前就把它拒绝了，工具根本没被调用。这正好看出两道关：格式由 SDK 把关，"病存不存在"由工具把关

### 面试怎么说
> "A `.mcp.json` in the repository gives everyone who opens the project the same tools in Claude Code; any other host takes the same launch command, and every call is traced."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 MCP 服务 | 官方 SDK 的 `MCPServer`，几十行 | 不自己写协议；服务层不加逻辑 |
| 2 从外面连上 | stdio：应用启动 `raredx mcp` | 隔一个进程、一层协议，结果不变 |
| 3 说明书 | 从工具代码自动生成 | 说明和代码永远一致 |
| 4 错误 | 坏参数先拒绝；"病不存在"告诉模型怎么改 | 模型能自己纠正 |
| 5 接 Claude Code | `.mcp.json` 进 git；调用记进 MLflow | 团队打开项目就能用，用了有记录 |

**一句话**：三个工具现在有了标准接口（MCP）；Claude Code、Claude Desktop、LangGraph 写的 agent，都能用同一条命令接上，看到同一份自动生成的说明书。

## 面试题

1. **What is MCP and why use it?**
   > "The Model Context Protocol is an open standard for exposing tools to AI applications. One server works with every MCP host — Claude Code, Claude Desktop, IDEs, LangGraph — instead of an adapter per framework."

2. **How do you keep tool documentation and behaviour in sync?**
   > "The schema is generated from the function's signature and docstring by the SDK; there is no second copy to drift."

3. **How should a tool report errors to a model?**
   > "As a result the model reads, with what to do next — 'use an id that rank_diseases returned' — for errors it can fix; unexpected exceptions stay in the server log so internals don't leak."

4. **stdio or HTTP?**
   > "stdio for a local tool a host launches per user — no port, nothing leaves the machine; Streamable HTTP when one deployed server serves many clients."

**下一章 14 · 固定流程：工具查完，交给大模型判断**：先不让大模型自己决定查什么，而是按固定顺序用工具查好，把结果交给它判断、写理由。这是计划里的对照组"固定流程"，后面的 agent 要和它比。这一章会调大模型，花几块钱，跑之前先跟你确认。